# THE ESPRESSO MODEL RELAY

**An illustrative, assumption-rich linked pull across separate models. This is not a validated coupled simulation.**

> One hypothetical espresso pull, passed from model to model. At each station we show what the model calculates, what is handed forward, what had to be assumed, and what the result might mean for the liquid that reaches the cup.

**Important — this is an illustrative model relay.**

Puckworks' components come from different papers, machines, coffees, geometries, definitions, and evidence levels. This notebook deliberately passes selected outputs between them to demonstrate possible platform capabilities. Some handoffs are direct, some use documented adapters, and some require substantial assumptions. The complete relay has **not** been validated as one scientific model. Treat its outputs as educational model lenses, not as a prediction of your espresso or its flavour.

This is **not** a digital twin, validated coupled model, universal simulator, recipe optimizer, or taste predictor. It runs **privately** in your Colab runtime (`LOCAL_PRIVATE`); that does not make any model cleared for public hosting.

**Development preview (0.4.0.dev0)** — not the v0.3.0 public release. This is a *separate* experience from the Full Laboratory Tour, which remains available and unchanged.

## 1. Set up (runs automatically — nothing to type)

In [ ]:
# This runs automatically — you do NOT type anything here.
# DEVELOPMENT PREVIEW (0.4.0.dev0): installs an EXACT, immutable pinned commit (no mutable 'main').
import os, subprocess, sys, hashlib

PIN_COMMIT = "3522c2409fe8f903b9f17cec9870f7b0a2d3f431"
INSTALL_SOURCE = f"git+https://github.com/trbrewer/puckworks@{PIN_COMMIT}"

override = os.environ.get("PUCKWORKS_WHEEL", "").strip()
if override:
    if not os.path.isfile(override):
        raise SystemExit("PUCKWORKS_WHEEL is set but is not a file: " + override)
    print("Hermetic mode — installing local candidate wheel")
    print("  wheel sha256:", hashlib.sha256(open(override, "rb").read()).hexdigest())
    INSTALL_SOURCE = override
else:
    print("DEVELOPMENT PREVIEW — installing pinned commit", PIN_COMMIT)

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "puckworks"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", INSTALL_SOURCE], check=True)
print("Install complete. install_source =", INSTALL_SOURCE)

if "puckworks" in sys.modules:
    raise SystemExit(
        "An older puckworks was already loaded in this Colab session. Fix: menu Runtime -> Restart "
        "runtime (or Disconnect and delete runtime), then Runtime -> Run all.")

## 2. Environment check

In [ ]:
import puckworks
print("puckworks version :", puckworks.__version__)
print("pinned commit     :", PIN_COMMIT)
assert puckworks.__version__ == "0.4.0.dev0", puckworks.__version__
try:
    from puckworks.product import linked_pull  # noqa: F401
except ImportError:
    raise SystemExit(
        "This runtime still has an OLDER puckworks without the Espresso Model Relay. Fix: menu "
        "Runtime -> Restart runtime (or Disconnect and delete runtime), then Runtime -> Run all.")
print("Espresso Model Relay available: yes")

## 3. How to read the relay

Each hand-off between models is drawn and labelled by kind:

- **solid line** — direct model output
- **dashed line** — documented adapter (units/basis converted, formula recorded)
- **dotted line** — illustrative assumption (a substantial assumed bridge)
- **thin grey line** — shared input or diagnostic only
- **branch line** — an alternative model of the same thing
- **blocked** — shown but never run (rights)

The real espresso processes overlap in time. The sequence below is the order of the educational/computational relay, **not** a claim that wetting finishes before extraction begins.

In [ ]:
#@title Your pull { display-mode: "form", run: "auto" }
#@markdown Fill in the short form, then run the single **Run the Espresso Model Relay** cell below.
dose_g = 20.0 #@param {type:"number"}
target_beverage_g = 40.0 #@param {type:"number"}
grind_setting = 1.7 #@param {type:"number"}
target_pressure_bar = 9.0 #@param {type:"number"}
brew_temperature_c = 93.0 #@param {type:"number"}
heterogeneity = "moderate" #@param ["low", "moderate", "high"]
execution_mode = "fast linked pull" #@param ["fast linked pull", "extended pore-scale relay"]

### Advanced assumptions used by this relay
This relay stipulates a tamped-puck connected porosity, a water-accessible porosity kept separate from it, a pump-to-bed line loss, and a basket area. They are listed in the assumption ledger near the bottom and in every downloaded report. You do not need to set them.

## 4. Run the Espresso Model Relay

Press the play button on the cell below. Everything runs **privately** in this Colab runtime (`LOCAL_PRIVATE`). Fast mode is a few seconds after install; the extended pore-scale relay takes materially longer.

In [ ]:
#@title ▶ Run the Espresso Model Relay { display-mode: "form" }
from puckworks.product.linked_pull import RelayRequest, execute_illustrative_linked_pull
mode = "extended" if execution_mode.startswith("extended") else "fast"
request = RelayRequest(dose_g=float(dose_g), target_beverage_g=float(target_beverage_g),
                       grind_setting=float(grind_setting), pressure_bar=float(target_pressure_bar),
                       brew_temperature_c=float(brew_temperature_c), heterogeneity=heterogeneity,
                       mode=mode)
print("Running the Espresso Model Relay across the espresso stages...", "(extended: slower)" if mode=="extended" else "(a few seconds)")
result = execute_illustrative_linked_pull(request, manifest_id="illustrative_linked_pull_v1",
                                          execution_context="LOCAL_PRIVATE", mode=mode)
c = result["counts"]
print("Executed", c["components_executed"], "components ·", c["cross_component_handoffs"], "hand-offs ·",
      c["assumptions_introduced"], "named assumptions.")

## 5. The scenario, at a glance, and the chain map

In [ ]:
try:
    get_ipython().run_line_magic("config", "InlineBackend.figure_format = 'retina'")
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass
from IPython.display import display, Markdown
from puckworks.product import linked_pull_display as D
display(Markdown("> _" + D.PERMANENT_WARNING + "_"))
display(Markdown(D.counts_summary(result)))
display(Markdown("### Chain map"))
print(D.chain_map_text(result))

## 6. The stations, in relay order

In [ ]:
from puckworks.product.linked_pull_figures import figures_by_owner
import matplotlib.pyplot as _plt
# figures draw purely from the completed result (zero model calls) and render beneath their OWNER
_figs = figures_by_owner(result)
for stage in result["stages"]:
    if stage["component_id"] == "recipe":
        continue
    blocks = D.station_blocks(stage)
    display(Markdown(blocks[0]))
    for b in blocks[1:3]:
        display(Markdown(b))
    for _f in _figs.get(stage["component_id"], []):
        display(_f.figure); _plt.close(_f.figure)
    for b in blocks[3:]:
        display(Markdown(b))
    display(Markdown("\n---\n"))

## 7. One pull, several readings

In [ ]:
for b in D.cup_dashboard_blocks(result):
    display(Markdown(b))

## 8. Assumption ledger

In [ ]:
display(Markdown("Every assumed hand-off, named and downloadable — not buried in a disclaimer."))
for a in result["assumptions"]:
    display(Markdown(f"- **{a['assumption_id']} ({a['category']}).** {a['statement']}"))

## 9. What would need to be measured to validate this relay?

In [ ]:
for a in result["assumptions"]:
    display(Markdown(f"- **{a['assumption_id']}** — {a['validation_needed']}"))

## 10. Download the full provenance-bearing report

In [ ]:
from puckworks.product.linked_pull_records import canonical_json_text
open("illustrative_linked_pull_v1.json", "w").write(canonical_json_text(result))
open("illustrative_linked_pull_v1.md", "w").write(D.relay_markdown(result))
print("Wrote illustrative_linked_pull_v1.json and .md")
# user-INITIATED download links (NO automatic download prompt)
try:
    from IPython.display import FileLink, display as _dl
    _dl(FileLink("illustrative_linked_pull_v1.json"))
    _dl(FileLink("illustrative_linked_pull_v1.md"))
except Exception:
    print("(files written to the working directory; open them from the file browser)")

## 11. Technical provenance

In [ ]:
display(Markdown("<details><summary><strong>Run provenance</strong></summary>\n\n"
    + f"- manifest: `{result['manifest_id']}`\n"
    + f"- schema: {result['schema_version']} · contract schema: {result['contract_schema_version']}\n"
    + f"- source commit: `{result['source_commit'][:12]}`\n"
    + f"- execution context: {result['execution_context']} · mode: {result['mode']}\n"
    + f"- model_output_hash: `{result['model_output_hash'][:16]}`\n"
    + f"- artifact_hash: `{result['artifact_hash'][:16]}`\n"
    + "- " + result['hash_note'] + "\n\n</details>"))
_ver = puckworks.__version__
print(f"LINKED_PULL_RELAY_COMPLETE: {_ver} · manifest={result['manifest_id']} · "
      f"context=LOCAL_PRIVATE · mode={result['mode']} · "
      f"executed={result['counts']['components_executed']} · "
      f"model_output_hash={result['model_output_hash']}")